In [2]:
import pandas as pd
import numpy as np

orders = pd.read_csv("./data/raw/orders.csv")
order_items = pd.read_csv("./data/raw/order_items.csv")
products = pd.read_csv("./data/raw/products.csv")
customers = pd.read_csv("./data/raw/customers.csv")

print("orders")
display(orders.head())

print("order_items")
display(order_items.head())

print("products")
display(products.head())

print("customers")
display(customers.head())

orders


,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2026-05-07,card,completed
1,2,77,2025-07-23,naver_pay,cancelled
2,3,138,2025-11-19,bank_transfer,cancelled
3,4,57,2026-01-30,kakao_pay,cancelled
4,5,125,2025-12-21,card,cancelled


order_items


,order_item_id,order_id,product_id,quantity,unit_price
0,1,1,100,3,102000
1,2,1,87,5,25000
2,3,1,7,3,142000
3,4,1,9,3,193000
4,5,2,72,4,189000


products


,product_id,product_name,category,price
0,1,전자기기 상품 001,전자기기,160000
1,2,도서 상품 002,도서,34000
2,3,전자기기 상품 003,전자기기,152000
3,4,생활용품 상품 004,생활용품,70000
4,5,식품 상품 005,식품,186000


customers


,customer_id,name,gender,age,city,signup_date
0,1,김수민,F,19,광주,2024-06-19
1,2,김정호,F,32,대구,2025-11-02
2,3,이경수,F,61,성남,2024-06-12
3,4,조영호,F,55,울산,2026-04-13
4,5,이예원,F,19,부산,2024-09-13


In [3]:
# 원본 구조 Evidence를 만든다

# raw_data = [customers, order_items, products, orders]
raw_data = {"customers" : customers, "order_items" : order_items,
             "products" : products, "orders" : orders}
print(raw_data)

{'customers':      customer_id name gender  age city signup_date
0              1  김수민      F   19   광주  2024-06-19
1              2  김정호      F   32   대구  2025-11-02
2              3  이경수      F   61   성남  2024-06-12
3              4  조영호      F   55   울산  2026-04-13
4              5  이예원      F   19   부산  2024-09-13
..           ...  ...    ...  ...  ...         ...
145          146  김숙자      M   61   성남  2025-12-23
146          147  이정남      M   19   부산  2025-02-11
147          148  오도현      M   29   고양  2026-06-15
148          149  김정자      M   20   부산  2024-10-19
149          150  조미영      M   40   대전  2025-12-04

[150 rows x 6 columns], 'order_items':      order_item_id  order_id  product_id  quantity  unit_price
0                1         1         100         3      102000
1                2         1          87         5       25000
2                3         1           7         3      142000
3                4         1           9         3      193000
4                5 

In [5]:
summary_list = []
for name, frame in raw_data.items():
    #print(name)
    info = {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing_values": frame.isna().sum().sum(),
        "duplicated_rows": frame.duplicated().sum() 
    }

    summary_list.append(info)

In [6]:
summary_list

[{'dataset': 'customers',
  'rows': 150,
  'columns': 6,
  'missing_values': np.int64(0),
  'duplicated_rows': np.int64(0)},
 {'dataset': 'order_items',
  'rows': 764,
  'columns': 5,
  'missing_values': np.int64(0),
  'duplicated_rows': np.int64(0)},
 {'dataset': 'products',
  'rows': 100,
  'columns': 4,
  'missing_values': np.int64(0),
  'duplicated_rows': np.int64(0)},
 {'dataset': 'orders',
  'rows': 300,
  'columns': 5,
  'missing_values': np.int64(0),
  'duplicated_rows': np.int64(0)}]

In [8]:

from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    validate_relationships,
)

processed_data = preprocess_sales_data(raw_data)
preprocessing_comparison = compare_shapes(
    raw_data,
    processed_data,
)

relationship_checks = validate_relationships(
    processed_data
)

display(preprocessing_comparison)
display(relationship_checks)

,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,764,5,764,6
2,orders,300,5,300,7
3,products,100,4,100,4


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


In [9]:
from src.preprocessing import compare_shapes, preprocess_sales_data

processed_data = preprocess_sales_data(raw_data)
preprocessing_comparison = compare_shapes(raw_data, processed_data)

preprocessing_comparison

,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,764,5,764,6
2,orders,300,5,300,7
3,products,100,4,100,4


In [11]:
print(raw_data["order_items"].columns)
print(processed_data["order_items"].columns)

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price'], dtype='str')
Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'line_total'],
      dtype='str')


In [12]:
print(processed_data["orders"].head())

   order_id  customer_id order_date payment_method order_status order_month  \
0         1          123 2026-05-07           card    completed     2026-05   
1         2           77 2025-07-23      naver_pay    cancelled     2025-07   
2         3          138 2025-11-19  bank_transfer    cancelled     2025-11   
3         4           57 2026-01-30      kakao_pay    cancelled     2026-01   
4         5          125 2025-12-21           card    cancelled     2025-12   

  order_dayofweek  
0        Thursday  
1       Wednesday  
2       Wednesday  
3          Friday  
4          Sunday  


In [14]:
print(processed_data["orders"]["order_dayofweek"].value_counts())

order_dayofweek
Saturday     50
Tuesday      47
Monday       47
Wednesday    44
Sunday       43
Friday       35
Thursday     34
Name: count, dtype: int64


In [17]:
key_map ={
    "customers": "customer_id",
    "products" : "product_id",
    "orders" : "order_id",
    "order_items" : "order_item_id",
}

In [19]:
pk_checks = []

for dataset, key in key_map.items():
    frame = processed_data[dataset]
    missing_count = int(frame[key].isna().sum())
    duplicated_count = int(frame[key].duplicated().sum())
    status = ""
    if missing_count == 0 and duplicated_count == 0 :
        status = "PASS"
    else:
        status = "FAIL"

    pk_checks.append(
        {
            "dataset": dataset,
            "key": key,
            "missing_count": missing_count,
            "duplicated_count": duplicated_count,
            "status": status,
        }
    )

pk_checks

[{'dataset': 'customers',
  'key': 'customer_id',
  'missing_count': 0,
  'duplicated_count': 0,
  'status': 'PASS'},
 {'dataset': 'products',
  'key': 'product_id',
  'missing_count': 0,
  'duplicated_count': 0,
  'status': 'PASS'},
 {'dataset': 'orders',
  'key': 'order_id',
  'missing_count': 0,
  'duplicated_count': 0,
  'status': 'PASS'},
 {'dataset': 'order_items',
  'key': 'order_item_id',
  'missing_count': 0,
  'duplicated_count': 0,
  'status': 'PASS'}]

In [ ]:

order_sales = order_items.merge(

    orders[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
        ]
    ],

    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,

)

In [24]:
print(order_sales.head())

   order_item_id  order_id  product_id  quantity  unit_price  customer_id  \
0              1         1         100         3      102000          123   
1              2         1          87         5       25000          123   
2              3         1           7         3      142000          123   
3              4         1           9         3      193000          123   
4              5         2          72         4      189000           77   

   order_date order_status _merge  
0  2026-05-07    completed   both  
1  2026-05-07    completed   both  
2  2026-05-07    completed   both  
3  2026-05-07    completed   both  
4  2025-07-23    cancelled   both  


In [26]:
print("병합 전 행 수 : ", len(order_items))
print("병합 후 행 수 : ", len(order_sales))
order_sales["_merge"].value_counts(dropna=False)

병합 전 행 수 :  764
병합 후 행 수 :  764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [28]:
expected_line_total = order_items["quantity"] * order_items["unit_price"]
expected_line_total

0      306000
1      125000
2      426000
3      579000
4      756000
        ...  
759    112000
760    696000
761    378000
762    700000
763    160000
Length: 764, dtype: int64

In [30]:
if "line_total" in order_items.columns:
    order_items["line_total"] = expected_line_total

order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
dtypes: int64(5)
memory usage: 30.0 KB


In [31]:
completed_order_sales = order_sales.loc[
    order_sales["order_status"].eq("completed")
]

print(completed_order_sales.head())
print(completed_order_sales["order_status"].value_counts())

    order_item_id  order_id  product_id  quantity  unit_price  customer_id  \
0               1         1         100         3      102000          123   
1               2         1          87         5       25000          123   
2               3         1           7         3      142000          123   
3               4         1           9         3      193000          123   
12             13         6          83         3       24000           87   

    order_date order_status _merge  
0   2026-05-07    completed   both  
1   2026-05-07    completed   both  
2   2026-05-07    completed   both  
3   2026-05-07    completed   both  
12  2026-03-21    completed   both  
order_status
completed    474
Name: count, dtype: int64


In [32]:
print("전체 주문 건수 : ", len(order_sales))
print("전체 주문 건수(completed) : ", len(completed_order_sales))
print("전체 주문 건수(completed 외) : ", len(order_sales) - len(completed_order_sales))

전체 주문 건수 :  764
전체 주문 건수(completed) :  474
전체 주문 건수(completed 외) :  290


In [33]:
order_sales["order_status"].value_counts()

order_status
completed    474
cancelled    162
refunded     128
Name: count, dtype: int64

In [4]:
completed_order_sales = order_sales[
    order_sales["order_status"] == "completed"
]

category_sales = (
    completed_order_sales
    .groupby("order_status", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )

)

NameError: name 'order_sales' is not defined

In [35]:
print(completed_order_sales.columns)
print(products.head())

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'customer_id', 'order_date', 'order_status', '_merge'],
      dtype='str')
   product_id product_name category   price
0           1  전자기기 상품 001     전자기기  160000
1           2    도서 상품 002       도서   34000
2           3  전자기기 상품 003     전자기기  152000
3           4  생활용품 상품 004     생활용품   70000
4           5    식품 상품 005       식품  186000
